# Module 03: Order Items EDA & Cleaning

### 1. Load Raw Data (Bronze Layer)

In [1]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("../data")
RAW_DATA_PATH = DATA_DIR / "olist_order_items_dataset.csv"

# Ensure the file exists before loading
assert RAW_DATA_PATH.exists(), f"Source file not found at: {RAW_DATA_PATH.resolve()}"

order_items = pd.read_csv(RAW_DATA_PATH)
print(f"Loaded raw dataset shape: {order_items.shape}")
display(order_items.head(3))

Loaded raw dataset shape: (112650, 7)


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.9,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.9,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.0,17.87


### 2. Data Audit & Missing Values Check
Analyzing data types, missing value distribution, and column cardinality.

In [2]:
audit_df = pd.DataFrame(
    {
        "Data Type": order_items.dtypes,
        "Missing Values": order_items.isna().sum(),
        "Missing %": (order_items.isna().mean() * 100).round(2),
        "Unique Values": order_items.nunique(),
    }
)

display(audit_df)

,Data Type,Missing Values,Missing %,Unique Values
order_id,object,0,0.0,98666
order_item_id,int64,0,0.0,21
product_id,object,0,0.0,32951
seller_id,object,0,0.0,3095
shipping_limit_date,object,0,0.0,93318
price,float64,0,0.0,5968
freight_value,float64,0,0.0,6999


### 3. Data Type Optimization & Silver Layer Initialization

To transition our dataset from the Bronze layer to the Silver layer, we enforce robust data types:
- **Timestamps (`datetime`):** Cast `shipping_limit_date` to enable precise temporal filtering, chronological validations, and downstream lead-time calculations.
- **Identifiers (`string`):** Standardize `order_id`, `product_id`, and `seller_id` into explicit string types for reliable relational joins and cross-table data integrity.

In [3]:
order_items_clean = order_items.copy()

order_items_clean["shipping_limit_date"] = pd.to_datetime(order_items_clean["shipping_limit_date"])

order_items_clean["order_id"] = order_items_clean["order_id"].astype("string")
order_items_clean["product_id"] = order_items_clean["product_id"].astype("string")
order_items_clean["seller_id"] = order_items_clean["seller_id"].astype("string")

display(order_items_clean.dtypes.to_frame(name="Silver Layer Data Type"))

,Silver Layer Data Type
order_id,string[python]
order_item_id,int64
product_id,string[python]
seller_id,string[python]
shipping_limit_date,datetime64[ns]
price,float64
freight_value,float64


### 4. Data Integrity Checks & Anomaly Detection

Ensuring logical consistency, validating business rules, and inspecting value distributions to isolate structural anomalies and outliers:
- **Price Validation:** Screening for negative or zero values in item prices (`price`).
- **Freight Value Check:** Verifying that shipping costs (`freight_value`) contain no negative anomalies.
- **Temporal Bounds Validation:** Checking for shipping dates (`shipping_limit_date`) falling outside the historical operational window (2016–2020).
- **Composite Key Uniqueness:** Verifying the integrity of primary keys by checking for duplicate `order_id` and `order_item_id` pairs.

In [4]:
audit_checks = {
    "Negative or zero price":
        order_items["price"] <= 0,
    "Negative freight value":
        order_items["freight_value"] < 0,
    "Shipping date out of historical bounds":
        (order_items["shipping_limit_date"] < "2016-01-01") | (order_items["shipping_limit_date"] > "2020-01-01"),
    "Exact duplicate rows":
        order_items.duplicated()
}

for label, condition in audit_checks.items():
    print(f"{label}: {condition.sum()}")

dup_composite = order_items.duplicated(subset=["order_id", "order_item_id"]).sum()
print(f"Duplicate order_id + order_item_id pairs: {dup_composite}")

Negative or zero price: 0
Negative freight value: 0
Shipping date out of historical bounds: 4
Exact duplicate rows: 0
Duplicate order_id + order_item_id pairs: 0


In [5]:
valid_dates_mask = (
    (order_items_clean["shipping_limit_date"] >= "2016-01-01") & 
    (order_items_clean["shipping_limit_date"] <= "2020-01-01")
)
order_items_clean = order_items_clean[valid_dates_mask]

### 5. Silver Layer Validation & Export
Comparing Bronze vs. Silver metrics (shape, types, and record count) and persisting the dataset to Parquet.

In [6]:
print("Pipeline Summary: Bronze vs Silver Layer")
print(f"Original shape (Bronze): {order_items.shape}")
print(f"Cleaned shape (Silver): {order_items_clean.shape}")
print(f"Total rows dropped: {len(order_items) - len(order_items_clean):,}")

# Side-by-side comparison of data types
dtypes_comparison = pd.DataFrame({
    'Bronze (Raw)': order_items.dtypes,
    'Silver (Clean)': order_items_clean.dtypes
})

print("Data Types Transformation:")
display(dtypes_comparison)

Pipeline Summary: Bronze vs Silver Layer
Original shape (Bronze): (112650, 7)
Cleaned shape (Silver): (112646, 7)
Total rows dropped: 4
Data Types Transformation:


,Bronze (Raw),Silver (Clean)
order_id,object,string[python]
order_item_id,int64,int64
product_id,object,string[python]
seller_id,object,string[python]
shipping_limit_date,object,datetime64[ns]
price,float64,float64
freight_value,float64,float64


## 6. Data Quality Assertions & Export

Before persisting the clean dataset to the Silver layer, we execute automated data quality checks tailored to transactional integrity and financial metrics:

* **Composite Key Integrity:** Ensure the combination of `order_id` and `order_item_id` is entirely unique and free of missing values.
* **Mandatory Attributes:** Verify critical fields (`product_id`, `seller_id`, `shipping_limit_date`) are fully populated.
* **Financial Constraints:** Validate that item `price` and `freight_value` contain no negative or zero anomalies.
* **Temporal Bounds Validity:** Confirm all `shipping_limit_date` values fall strictly within the realistic historical operational window (2016–2020).
* **Dataset Validity:** Ensure the resulting dataset is non-empty before writing to storage.

In [7]:
assert order_items_clean.duplicated(subset=['order_id', 'order_item_id']).sum() == 0, "Assertion failed: Duplicate composite key (order_id, order_item_id) found"
assert order_items_clean['order_id'].notna().all(), "Assertion failed: NULL values found in order_id"
assert order_items_clean['order_item_id'].notna().all(), "Assertion failed: NULL values found in order_item_id"

critical_cols = ['product_id', 'seller_id', 'shipping_limit_date']
for col in critical_cols:
    assert order_items_clean[col].notna().all(), f"Assertion failed: NULL values found in mandatory column '{col}'"

assert (order_items_clean['price'] > 0).all(), "Assertion failed: Non-positive price values found"
assert (order_items_clean['freight_value'] >= 0).all(), "Assertion failed: Negative freight values found"

valid_shipping_dates = (
    (order_items_clean['shipping_limit_date'] >= "2016-01-01") & 
    (order_items_clean['shipping_limit_date'] <= "2020-01-01")
)
assert valid_shipping_dates.all(), "Assertion failed: Shipping limit dates out of historical bounds found"

assert len(order_items_clean) > 0, "Assertion failed: Cleaned dataset is empty"

print("All quality checks passed successfully. Saving to Silver layer.")
order_items_clean.to_parquet("../data/order_items_clean.parquet", index=False)
print("Order Items dataset successfully saved to Parquet.")

All quality checks passed successfully. Saving to Silver layer.
Order Items dataset successfully saved to Parquet.
